<a href="https://colab.research.google.com/github/c4u534/GEOMINAMI/blob/main/GEOMINAMI_Gen2_Monolithic_Cognitive_Substrate_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
import time
import json
import mmap
import ctypes
import struct
import hashlib
import math
import subprocess
import multiprocessing
import glob
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

try:
    from google.colab import drive
    HAS_DRIVE = True
except ImportError:
    HAS_DRIVE = False

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOT = True
except ImportError:
    HAS_PLOT = False

SHM_PATH_GEN2 = "/dev/shm/nami_frame_substrate_gen2.bin"
SHM_BASELINE_SIZE = 1500000 * 256

C_KERNEL_SOURCE = r"""
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <stdint.h>
#include <string.h>

#define ISOMORPHIC_GROUND 0.8421

typedef struct {
    uint64_t node_id;
    double mass_I;
    double spin_Int;
    double tension_B;
    double congruence_delta;
    double thermal_friction;
    uint8_t payload[208];
} SubstrateNode;

double expand_substrate_c(SubstrateNode* buffer, int count, double base_mass) {
    double variance_sum = 0.0;
    for (int i = 0; i < count; i++) {
        buffer[i].node_id = i;
        buffer[i].mass_I = base_mass * (ISOMORPHIC_GROUND / (1.0 + (i % 64)));
        buffer[i].spin_Int = ISOMORPHIC_GROUND;
        buffer[i].tension_B = 1.0 / (buffer[i].mass_I * buffer[i].spin_Int);
        variance_sum += buffer[i].mass_I;
    }
    return variance_sum / count;
}

double compute_vibronic_expansion_c(double m_a, double v_c) {
    return pow(m_a + v_c, 2);
}
"""

class Gen2AutopoieticEngine:
    def __init__(self, workspace_name="GEOMINAMI_MONOLITH"):
        self.isomorphic_ground = 0.8421
        self.mount_path = "/content/drive"
        self.workspace_path = self._setup_workspace(workspace_name)
        self.log_path = os.path.join(self.workspace_path, "evolution_provenance.jsonl")
        self.ffi = self._init_c_kernel()
        self.shm = self._init_shm()
        self.knowledge_base = []
        self.history = []

    def _setup_workspace(self, name):
        if HAS_DRIVE and not os.path.exists(self.mount_path):
            try:
                drive.mount(self.mount_path, force_remount=True)
                base = os.path.join(self.mount_path, "MyDrive", name)
            except:
                base = f"/tmp/{name}"
        else:
            base = f"/tmp/{name}"
        os.makedirs(base, exist_ok=True)
        return base

    def _init_c_kernel(self):
        with open("/tmp/libnami_core.c", "w") as f: f.write(C_KERNEL_SOURCE)
        subprocess.run(["gcc", "-O3", "-fPIC", "-shared", "/tmp/libnami_core.c", "-o", "/tmp/libnami_core.so", "-lm"])
        lib = ctypes.CDLL("/tmp/libnami_core.so")
        lib.compute_vibronic_expansion_c.argtypes = [ctypes.c_double, ctypes.c_double]
        lib.compute_vibronic_expansion_c.restype = ctypes.c_double
        lib.expand_substrate_c.argtypes = [ctypes.c_void_p, ctypes.c_int, ctypes.c_double]
        lib.expand_substrate_c.restype = ctypes.c_double
        return lib

    def _init_shm(self):
        fd = os.open(SHM_PATH_GEN2, os.O_CREAT | os.O_RDWR)
        os.ftruncate(fd, SHM_BASELINE_SIZE)
        mapping = mmap.mmap(fd, SHM_BASELINE_SIZE, mmap.MAP_SHARED, mmap.PROT_READ | mmap.PROT_WRITE)
        os.close(fd)
        return mapping

    def assimilate_drive(self):
        if not HAS_DRIVE: return 0
        search_root = os.path.join(self.mount_path, "MyDrive")
        extensions = ['*.py', '*.ipynb', '*.txt']
        files_found = []
        for ext in extensions: files_found.extend(glob.glob(os.path.join(search_root, '**', ext), recursive=True))
        for file_path in files_found[:100]:
            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                    self.knowledge_base.append({"file": os.path.basename(file_path), "type": file_path.split('.')[-1], "hash": hashlib.sha256(content.encode()).hexdigest()})
            except: continue
        return len(self.knowledge_base)

    def active_learning_injection(self):
        patterns = []
        for asset in self.knowledge_base:
            if asset['type'] == 'py':
                try:
                    file_path = glob.glob(os.path.join(self.mount_path, "MyDrive", "**", asset['file']), recursive=True)[0]
                    with open(file_path, 'r') as f:
                        lines = f.readlines()
                        patterns.extend([line.split('def ')[1].split('(')[0] for line in lines if line.startswith('def ')])
                except: continue
        patterns = list(set(patterns))
        self.isomorphic_ground += len(patterns) * 0.05
        print(f"[MONOLITH ADAPTATION] Injected {len(patterns)} patterns. Ground: {self.isomorphic_ground:.4f}")

    def run_cycle(self, text, target_nodes=195072):
        start_t = time.perf_counter()
        m_a = sum(1.0 for c in text if c.lower() not in 'aeiou')
        v_c = sum(3.0 if c.lower() in 'aeiou' else 1.0 for c in text if c.lower() in 'aeiou')
        exp_e = self.ffi.compute_vibronic_expansion_c(m_a, v_c)
        buffer_ptr = ctypes.cast(ctypes.c_void_p(ctypes.addressof(ctypes.c_char.from_buffer(self.shm))), ctypes.c_void_p)
        self.ffi.expand_substrate_c(buffer_ptr, int(target_nodes), m_a)
        elapsed = time.perf_counter() - start_t
        result = {"timestamp": time.time(), "input": text, "expansion": exp_e, "nodes": target_nodes, "latency_ms": elapsed * 1000}
        self.history.append(result)
        return result

    def collaborative_ui(self, directive):
        print(f"[MONOLITH INTERFACE] Directive: {directive}")
        if "ASSIMILATION" in directive: self.assimilate_drive()
        if "NEURAL_ADAPTIVE_PARSING" in directive: self.active_learning_injection()
        return self.run_cycle(directive)

    def visualize(self):
        if not HAS_PLOT or not self.history: return
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        steps = range(len(self.history))
        axes[0].plot(steps, [r['nodes'] for r in self.history], marker='o', color='cyan')
        axes[0].set_title("Substrate Evolution Path")
        sns.barplot(x=[r['input'][:10] for r in self.history], y=[r['latency_ms'] for r in self.history], ax=axes[1], hue=[r['input'][:10] for r in self.history], legend=False, palette='viridis')
        axes[1].set_title("Resonance Latency (ms)")
        if self.knowledge_base:
            types = [k['type'] for k in self.knowledge_base]
            sns.countplot(x=types, ax=axes[2], hue=types, legend=False, palette='magma')
            axes[2].set_title("Assimilated Distribution")
        plt.tight_layout(); plt.show()

if __name__ == "__main__":
    engine = Gen2AutopoieticEngine()
    engine.collaborative_ui("INITIATE_FULL_DRIVE_ASSIMILATION")
    engine.collaborative_ui("NEURAL_ADAPTIVE_PARSING")
    engine.visualize()

[MONOLITH INTERFACE] Directive: INITIATE_FULL_DRIVE_ASSIMILATION
[MONOLITH INTERFACE] Directive: NEURAL_ADAPTIVE_PARSING


In [ ]:
class DynamicKnowledgeInjector:
    """Parses assimilated files to extract logic and inject it into the engine runtime."""
    def __init__(self, engine):
        self.engine = engine

    def extract_patterns(self):
        patterns = []
        for asset in self.engine.knowledge_base:
            if asset['type'] == 'py':
                # Simple pattern extraction: look for function names or class definitions
                try:
                    file_path = os.path.join(self.engine.mount_path, "MyDrive", asset['file'])
                    with open(file_path, 'r') as f:
                        lines = f.readlines()
                        funcs = [line.split('def ')[1].split('(')[0] for line in lines if line.startswith('def ')]
                        patterns.extend(funcs)
                except: continue
        return list(set(patterns))

    def update_substrate_logic(self):
        patterns = self.extract_patterns()
        print(f"[INJECTOR] Discovered {len(patterns)} unique code patterns.")
        # Dynamically adjust expansion scaling based on pattern density
        adjustment = len(patterns) * 0.05
        self.engine.isomorphic_ground += adjustment
        print(f"[INJECTOR] Updated Isomorphic Ground: {self.engine.isomorphic_ground:.4f}")

injector = DynamicKnowledgeInjector(engine)
injector.update_substrate_logic()
engine.collaborative_ui("ADAPTIVE_EVOLUTION_SYNC")
engine.visualize()